# 🧠 CalRetail — Real-Time Agent Assist
## Goal
Retrieve similar solved tickets and standard operating procedures (SOP) to assist support agents.

## Algorithmic Explanation
**Semantic search using TF-IDF and Cosine Similarity**
1. Vectorize historic solved ticket logs.
2. Fit cosine similarities on query vectors.
3. Pull matched cases to extract corresponding SOP rules.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tickets = load_table('support_tickets')
resolved = tickets[tickets['status'] == 'Resolved'].copy().head(500) # subset for speed
resolved['description'] = resolved['description'].fillna('No description')

tfidf = TfidfVectorizer(max_features=500, stop_words='english')
matrix = tfidf.fit_transform(resolved['description'])
print("Solved tickets indexed successfully.")


In [ ]:
SOP_MAP = {
    "Return & Refund": "Cross check transaction date; if < 10 days, approve return labels.",
    "Wrong Item": "Cross check transaction date; if < 10 days, approve return labels.",
    "Delivery Delay": "Query warehouse shipments status and update shipping delivery schedule.",
    "Product Quality": "Request images of damage; issue replacement/refund SOP.",
    "Payment Problem": "Escalate to finance log; verify merchant gateway ID transaction.",
    "Account Issue": "Verify customer identity via OTP, reset password if requested.",
}
DEFAULT_SOP = "Initiate standardized customer service check-in."

KNOWLEDGE_MAP = {
    "Return & Refund": [
        {"title": "CalRetail Return & Exchange Policy SOP", "url": "https://kb.calretail.com/policies/returns"},
        {"title": "How to Process a Refund in POS", "url": "https://kb.calretail.com/sop/refunds"}
    ],
    "Wrong Item": [
        {"title": "CalRetail Return & Exchange Policy SOP", "url": "https://kb.calretail.com/policies/returns"},
        {"title": "How to Process a Refund in POS", "url": "https://kb.calretail.com/sop/refunds"}
    ],
    "Delivery Delay": [
        {"title": "Tracking Shipments via Delhivery API", "url": "https://kb.calretail.com/sop/delivery-tracking"},
        {"title": "Customer Communication Strategy for Delays", "url": "https://kb.calretail.com/sop/delay-communications"}
    ],
    "Product Quality": [
        {"title": "Product Quality Standards and Reporting", "url": "https://kb.calretail.com/sop/quality-inspection"}
    ],
    "Payment Problem": [
        {"title": "Payment Merchant Reconciliation Flow", "url": "https://kb.calretail.com/sop/payment-reconciliation"}
    ],
}
DEFAULT_KNOWLEDGE = [
    {"title": "General Customer Support Flow & Escalation SLA", "url": "https://kb.calretail.com/sop/general-escalations"}
]


def get_agent_assist(agent_query):
    query_vec = tfidf.transform([agent_query])
    sims = cosine_similarity(query_vec, matrix)[0]

    top_3_idx = sims.argsort()[-3:][::-1]
    matches = []
    for idx in top_3_idx:
        row = resolved.iloc[idx]
        matches.append({
            "ticket_id": row['ticket_id'],
            "description": row['description'],
            "category": row['category'],
            "similarity": round(float(sims[idx]), 3),
            "suggested_reply": SOP_MAP.get(row['category'], DEFAULT_SOP)
        })

    cat = matches[0]['category'] if matches else "General"
    recommended_sop = SOP_MAP.get(cat, DEFAULT_SOP)
    knowledge_articles = KNOWLEDGE_MAP.get(cat, DEFAULT_KNOWLEDGE)

    return {
        "query": agent_query,
        "matched_tickets": matches,        # old key for test compliance
        "suggested_responses": matches,    # new key
        "recommended_sop": recommended_sop,
        "knowledge_articles": knowledge_articles
    }

query = "My package has not delivered yet"
backend_res = get_agent_assist(query)
print("Agent Assist Output:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL SUPPORT AGENT HELPdesk ===")
print(f"Customer Question: '{query}'")
print(f"Suggested SOP: {backend_res['recommended_sop']}\n")
print("Top Similar Historic Tickets:")
for t in backend_res['matched_tickets']:
    print(f"  - Match [{t['category']}]: \"{t['description'][:60]}...\" (Match: {t['similarity']})")
